# The Superposition Hypothesis

**Survey Reference:** Section 6.2 - The Superposition Hypothesis

## Overview

The **Superposition Hypothesis** proposes that neural networks represent more features than they have dimensions by encoding features as nearly-orthogonal directions in activation space. This is a key insight for understanding why interpretability is hard.

### Key Ideas

- Networks can represent $m$ features in $n$ dimensions where $m >> n$
- Features are stored as almost-orthogonal vectors (with small interference)
- Sparse features can tolerate more interference than dense ones
- This creates **polysemanticity**: single neurons respond to multiple unrelated concepts

## Learning Objectives

1. Understand why superposition emerges (capacity vs. interference tradeoff)
2. Build toy models demonstrating superposition
3. Measure superposition in real models
4. Connect to sparse autoencoders as a solution

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from einops import rearrange, einsum

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. The Geometry of Superposition

### Johnson-Lindenstrauss Lemma

In high dimensions, random vectors are nearly orthogonal. For $m$ random unit vectors in $\mathbb{R}^n$:

$$\mathbb{E}[|v_i \cdot v_j|] \approx \sqrt{\frac{2}{\pi n}}$$

This means we can pack many more features than dimensions with small interference!

In [ ]:
# Demonstrate near-orthogonality of random vectors in high dimensions
def measure_random_vector_angles(n_dims, n_vectors, n_trials=100):
    """Measure average dot product between random unit vectors."""
    avg_dots = []
    for _ in range(n_trials):
        vectors = torch.randn(n_vectors, n_dims)
        vectors = vectors / vectors.norm(dim=1, keepdim=True)
        dots = vectors @ vectors.T
        # Exclude diagonal
        mask = ~torch.eye(n_vectors, dtype=bool)
        avg_dots.append(dots[mask].abs().mean().item())
    return np.mean(avg_dots)

# TODO: Plot interference vs dimension

## 2. Toy Model of Superposition

Following Anthropic's toy model, we train a simple autoencoder:

$$h = W^T x \quad \text{(encode to bottleneck)}$$
$$\hat{x} = \text{ReLU}(W h + b) \quad \text{(decode back)}$$

With $m$ input features and $n < m$ hidden dimensions.

In [ ]:
class ToyModel(nn.Module):
    """Toy model to study superposition."""
    
    def __init__(self, n_features, n_hidden):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n_features, n_hidden) * 0.1)
        self.b = nn.Parameter(torch.zeros(n_features))
    
    def forward(self, x):
        # Encode: project to hidden space
        h = x @ self.W  # [batch, n_hidden]
        # Decode: project back with ReLU
        x_hat = torch.relu(h @ self.W.T + self.b)
        return x_hat
    
    @property
    def feature_directions(self):
        """Get the direction each feature is encoded in."""
        return self.W / self.W.norm(dim=1, keepdim=True)

In [ ]:
def generate_sparse_data(n_samples, n_features, sparsity):
    """
    Generate sparse feature data.
    Each feature is active with probability (1 - sparsity).
    """
    # Feature values when active (uniform)
    values = torch.rand(n_samples, n_features)
    # Sparsity mask
    mask = torch.rand(n_samples, n_features) > sparsity
    return values * mask.float()

# TODO: Generate training data with varying sparsity

In [ ]:
def train_toy_model(n_features, n_hidden, sparsity, n_steps=10000):
    """Train toy model and return learned feature directions."""
    model = ToyModel(n_features, n_hidden).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # TODO: Training loop
    # Use importance-weighted MSE loss
    
    return model

# TODO: Train models with different sparsity levels

## 3. Visualizing Superposition

### Feature Geometry

With enough sparsity, features arrange into geometric structures (pentagons, etc.) that maximize orthogonality.

In [ ]:
def plot_feature_geometry(model, title="Feature Geometry"):
    """Visualize how features are arranged in hidden space."""
    W = model.W.detach().cpu().numpy()
    
    if W.shape[1] == 2:
        # 2D hidden space - plot directly
        plt.figure(figsize=(8, 8))
        for i in range(W.shape[0]):
            plt.arrow(0, 0, W[i, 0], W[i, 1], head_width=0.05, 
                     head_length=0.02, fc=f'C{i}', ec=f'C{i}')
        plt.xlim(-1.5, 1.5)
        plt.ylim(-1.5, 1.5)
        plt.gca().set_aspect('equal')
        plt.title(title)
        plt.grid(True, alpha=0.3)
        plt.show()
    else:
        print("Use PCA for higher-dimensional hidden spaces")

# TODO: Visualize feature arrangements for different sparsity levels

## 4. Measuring Superposition

### Interference Score

We can quantify superposition by measuring how much features interfere with each other.

In [ ]:
def compute_interference(model):
    """Compute feature interference matrix."""
    W = model.W
    W_normed = W / W.norm(dim=1, keepdim=True)
    interference = (W_normed @ W_normed.T) ** 2
    # Zero out diagonal
    interference = interference - torch.diag(torch.diag(interference))
    return interference.detach().cpu().numpy()

# TODO: Visualize interference patterns

## 5. Phase Transitions

Superposition exhibits phase transitions as sparsity changes:
- **Low sparsity:** Features represented 1:1 with neurons (no superposition)
- **High sparsity:** Many features packed into few neurons (heavy superposition)

In [ ]:
# TODO: Plot phase diagram of superposition vs. sparsity and feature importance

## 6. Implications for Interpretability

Superposition explains why:

1. **Neurons are polysemantic:** They respond to multiple unrelated features
2. **Linear probes work:** Features are still (approximately) linear directions
3. **SAEs are needed:** To decompose superposed representations into individual features

### Connection to Sparse Autoencoders

SAEs attempt to "undo" superposition by learning a larger dictionary of sparse features.

In [ ]:
# TODO: Preview of SAE approach (detailed in 03_feature_extraction/)

## Key Takeaways

1. **Superposition is a capacity optimization:** Networks pack more features than dimensions
2. **Sparsity enables superposition:** Rarely co-occurring features can share dimensions
3. **Interference is the cost:** Features slightly corrupt each other's signals
4. **Geometry matters:** Features arrange into structured patterns (polytopes)
5. **This explains polysemanticity:** Single neurons encode multiple features

## Next Steps

→ **03_privileged_access_hypothesis.ipynb**: When do neurons align with interpretable features?